# Taobots — Workshop Analysis

Loads the most recent workshop log and gives tick-by-tick visibility into a single bot's
organ integrity, storage, behaviour, movement, and metabolic balance.

**Log file expected in `../logs/`:**
- `<world>_workshop_<timestamp>.csv` — one row per simulated tick

Run the workshop first: `python main.py --workshop`

In [ ]:
import glob
import pathlib

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

LOGS = pathlib.Path('../logs')
WORLD = 'workshop'

plt.rcParams.update({
    'figure.facecolor': '#0a1a1a',
    'axes.facecolor':   '#0d2020',
    'axes.edgecolor':   '#304040',
    'axes.labelcolor':  '#b0c0b0',
    'xtick.color':      '#607060',
    'ytick.color':      '#607060',
    'text.color':       '#b0c0b0',
    'grid.color':       '#1e3030',
    'grid.linewidth':   0.5,
    'legend.facecolor': '#0d2020',
    'legend.edgecolor': '#304040',
})

# --- constants, imported rather than copied ---------------------------------
# Hardcoded copies drifted badly: this notebook's drain rates were both inverted
# (Earth/Wood swapped by E1) and 10x too small. Notebooks are outside ruff, mypy
# and CI, so nothing catches a stale copy — import from the source instead.
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from common import ELEMENT_COLOR, ELEMENT_LIST          # noqa: E402
from taobot_simple import (                              # noqa: E402
    DEFAULT_PARAMS,
    DERIVED_ORGANS,
    FIRE_LOCKOUT_THRESHOLD,
    ORGAN_STORAGE_DRAIN,
)

FLEE_THRESHOLD = DEFAULT_PARAMS['flee_earth_threshold']
STRUCTURAL = 'EARTH'                      # the death organ (AD-7). Was Wood before E1.
DERIVED = {e.name for e in DERIVED_ORGANS}

ELEMENT_COLORS = {e.name: '#%02X%02X%02X' % ELEMENT_COLOR[e] for e in ELEMENT_LIST}

STATE_COLORS = {
    'searching':  '#6090a0',
    'seeking':    '#00dc78',
    'collecting': '#00ff40',
    'fleeing':    '#dcdc00',
}

# Load most recent workshop log
ws_files = sorted(glob.glob(str(LOGS / f'{WORLD}_workshop_*.csv')))
if not ws_files:
    raise FileNotFoundError(f'No workshop logs found in {LOGS}. Run: python main.py --workshop')

df = pd.read_csv(ws_files[-1])
print(f'Loaded: {ws_files[-1]}')

# A workshop run that spans a death contains SEVERAL bots: target_population is 1, so a
# fresh one spawns with undamaged legs and full reserves. Plotted end to end that reads
# as a spontaneous recovery — integrity snapping back to 1.0 with nothing having repaired
# it. Keep the longest single life by default; FULL_RUN holds the spliced series.
FULL_RUN = df
if 'entity_id' in df.columns and df['entity_id'].nunique() > 1:
    lives = df.groupby('entity_id').size().sort_values(ascending=False)
    keep = lives.index[0]
    print(f'NOTE: this run spans {len(lives)} bots (ids {sorted(df["entity_id"].unique())}). '
          f'Keeping the longest life, bot {keep} ({lives.iloc[0]} ticks). '
          f'Use FULL_RUN to plot the spliced series instead.')
    df = df[df['entity_id'] == keep].reset_index(drop=True)
print(f'Ticks: {len(df)}  |  Archetype: {df["archetype"].iloc[0]}  |  Bot ID: {df["entity_id"].iloc[0]}')
df.head(3)

## 1. Organ integrity

In [ ]:
ELEMENTS = [e.name for e in ELEMENT_LIST]

fig, ax = plt.subplots(figsize=(14, 5))

for elem in ELEMENTS:
    col = f'organ_{elem}'
    lw = 2.2 if elem == STRUCTURAL else 1.2
    label = elem + (' (derived)' if elem in DERIVED else '')
    ax.plot(df['tick'], df[col], color=ELEMENT_COLORS[elem], linewidth=lw, label=label)

ax.axhline(FLEE_THRESHOLD, color='#ff500a', linewidth=0.8, linestyle='--',
           label=f'Earth flee threshold ({FLEE_THRESHOLD:.0f})')
ax.axhline(FIRE_LOCKOUT_THRESHOLD, color='#cccc00', linewidth=0.8, linestyle=':',
           label=f'Fire lockout ({FIRE_LOCKOUT_THRESHOLD:.0f})')

# Shade behaviour states along the x-axis
state_changes = df[df['behavior_state'] != df['behavior_state'].shift()]
for i, row in state_changes.iterrows():
    end_tick = df.loc[i+1, 'tick'] if i + 1 < len(df) else df['tick'].iloc[-1]
    ax.axvspan(row['tick'], end_tick,
               color=STATE_COLORS.get(row['behavior_state'], '#333'),
               alpha=0.08, linewidth=0)

ax.set_ylim(0, 105)
ax.set_ylabel('Organ value (0–100)')
ax.set_xlabel('Tick')
ax.set_title('Organ Integrity — tick by tick  (shaded by behaviour state)')
ax.legend(loc='upper right', ncol=4)
ax.grid(True)
plt.tight_layout()
plt.show()

## 2. Storage levels

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for elem in ELEMENTS:
    col = f'storage_{elem}'
    ax.plot(df['tick'], df[col], color=ELEMENT_COLORS[elem], linewidth=1.2, label=elem)

ax.set_ylabel('Storage level')
ax.set_xlabel('Tick')
ax.set_title('Storage Levels')
ax.legend(loc='upper right', ncol=5)
ax.grid(True)
plt.tight_layout()
plt.show()

## 3. Behaviour state timeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 1.6))

states = list(df['behavior_state'])
ticks  = list(df['tick'])

prev_state = states[0]
seg_start  = ticks[0]

for i in range(1, len(states)):
    if states[i] != prev_state or i == len(states) - 1:
        seg_end = ticks[i]
        ax.barh(0, seg_end - seg_start, left=seg_start, height=0.8,
                color=STATE_COLORS.get(prev_state, '#555'), align='center')
        prev_state = states[i]
        seg_start  = ticks[i]

legend_patches = [mpatches.Patch(color=c, label=s) for s, c in STATE_COLORS.items()]
ax.legend(handles=legend_patches, loc='upper right', ncol=4, fontsize=8)
ax.set_yticks([])
ax.set_xlabel('Tick')
ax.set_title('Behaviour State Timeline')
ax.set_xlim(ticks[0], ticks[-1])
plt.tight_layout()
plt.show()

# Breakdown table
state_counts = df['behavior_state'].value_counts()
state_pct    = (state_counts / len(df) * 100).round(1)
print(pd.DataFrame({'ticks': state_counts, 'pct': state_pct}).to_string())

## 4. Tick-by-tick resource intake and damage

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

bottoms = np.zeros(len(df))
for elem in ELEMENTS:
    col = f'intake_{elem}'
    vals = df[col].values
    ax.bar(df['tick'], vals, bottom=bottoms,
           color=ELEMENT_COLORS[elem], label=elem, width=1.0, alpha=0.9)
    bottoms += vals

ax2 = ax.twinx()
ax2.plot(df['tick'], df['tick_damage'], color='#dc3030',
         linewidth=1.0, linestyle='--', label='damage')
ax2.set_ylabel('Damage taken (per tick)', color='#dc3030')
ax2.tick_params(axis='y', labelcolor='#dc3030')
ax2.spines['right'].set_color('#dc3030')

ax.set_ylabel('Resources collected (per tick)')
ax.set_xlabel('Tick')
ax.set_title('Tick-by-tick Intake & Damage')
ax.legend(loc='upper left', ncol=5)
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

## 5. Position trace

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ticks_norm = (df['tick'] - df['tick'].min()) / max(df['tick'].max() - df['tick'].min(), 1)
cmap = plt.cm.cool

for i in range(len(df) - 1):
    ax.plot(
        df['x'].iloc[i:i+2], df['y'].iloc[i:i+2],
        color=cmap(ticks_norm.iloc[i]), linewidth=0.8, alpha=0.7,
    )

# Start / end markers
ax.scatter(df['x'].iloc[0],  df['y'].iloc[0],  color='#00ff80', s=60, zorder=5, label='start')
ax.scatter(df['x'].iloc[-1], df['y'].iloc[-1], color='#ff4040', s=60, zorder=5, label='end')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(df['tick'].min(), df['tick'].max()))
plt.colorbar(sm, ax=ax, label='tick')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Position Trace  (cool = early → warm = late)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f'Total distance moved: {df["distance_moved"].iloc[-1]:.1f} units')
print(f'Bounding box: x={df["x"].min():.1f}–{df["x"].max():.1f}  y={df["y"].min():.1f}–{df["y"].max():.1f}')

## 6. Metabolic balance

Per-tick intake vs. per-tick organ drain. Positive = surplus; negative = organ degrading.

Base drain rates per tick (FIRE, EARTH, WOOD, METAL only — Water is consumed by LegParts):
```
FIRE  0.0015    EARTH 0.0010    WOOD  0.0004    METAL 0.0002
```

In [ ]:
# Imported, not copied — the hardcoded version here was inverted AND 10x too small.
# WATER is absent by design: LegPart spends it, not an organ drain.
DRAIN_PER_TICK = dict(ORGAN_STORAGE_DRAIN)
METABOLIC_ELEMENTS = list(DRAIN_PER_TICK.keys())

fig, axes = plt.subplots(len(METABOLIC_ELEMENTS), 1, figsize=(14, 2.5 * len(METABOLIC_ELEMENTS)), sharex=True)

for ax, elem in zip(axes, METABOLIC_ELEMENTS):
    intake = df[f'intake_{elem}'].values
    demand = DRAIN_PER_TICK[elem]
    balance = intake - demand

    ax.axhline(0, color='white', linewidth=0.7, linestyle='--')
    ax.fill_between(df['tick'], balance, 0,
                    where=(balance >= 0), color=ELEMENT_COLORS[elem], alpha=0.6, label='surplus')
    ax.fill_between(df['tick'], balance, 0,
                    where=(balance < 0),  color='#800000',              alpha=0.6, label='deficit')
    ax.plot(df['tick'], balance, color=ELEMENT_COLORS[elem], linewidth=0.6)

    pct_deficit = (balance < 0).mean() * 100
    ax.set_ylabel(elem, rotation=0, labelpad=36)
    ax.set_title(f'{elem}  —  {pct_deficit:.0f}% of ticks in deficit', loc='left', fontsize=9)
    ax.grid(True)

axes[-1].set_xlabel('Tick')
fig.suptitle('Metabolic Balance per Element  (intake − drain)', y=1.01)
plt.tight_layout()
plt.show()

## 6.5 Leg state

Per-leg water reserve, structural integrity, and per-tick thrust. Detects water
starvation (reserve → 0) and integrity degradation before it shows up as slower movement.

In [ ]:
leg_cols = [c for c in df.columns if c.startswith('leg_')]
leg_ids = sorted({c.split('_')[1] for c in leg_cols})

if not leg_ids:
    print('No leg columns in this log — regenerate with updated workshop logger.')
else:
    n_legs = len(leg_ids)
    fig, axes = plt.subplots(n_legs, 3, figsize=(16, 3 * n_legs), squeeze=False)
    leg_color = ELEMENT_COLORS['WATER']

    for row_i, leg_i in enumerate(sorted(leg_ids, key=int)):
        reserve_col   = f'leg_{leg_i}_reserve'
        integrity_col = f'leg_{leg_i}_integrity'
        thrust_col    = f'leg_{leg_i}_thrust'

        ax = axes[row_i, 0]
        ax.plot(df['tick'], df[reserve_col], color=leg_color, linewidth=1.2)
        ax.set_title(f'Leg {leg_i} — Water Reserve', loc='left', fontsize=9)
        ax.set_ylabel('reserve')
        ax.grid(True)

        ax = axes[row_i, 1]
        ax.plot(df['tick'], df[integrity_col], color='#c0c060', linewidth=1.2)
        ax.set_ylim(0, 1.05)
        ax.axhline(1.0, color='#404040', linewidth=0.6, linestyle='--')
        ax.set_title(f'Leg {leg_i} — Structural Integrity', loc='left', fontsize=9)
        ax.set_ylabel('integrity')
        ax.grid(True)

        ax = axes[row_i, 2]
        ax.plot(df['tick'], df[thrust_col], color='#60c080', linewidth=1.2)
        ax.axhline(0, color='#404040', linewidth=0.6)
        ax.set_title(f'Leg {leg_i} — Thrust', loc='left', fontsize=9)
        ax.set_ylabel('thrust')
        ax.grid(True)

    for ax in axes[-1]:
        ax.set_xlabel('Tick')

    plt.tight_layout()
    plt.show()

    # Summary table
    print('=== LEG STATE SUMMARY (final tick) ===')
    for leg_i in sorted(leg_ids, key=int):
        r = df[f'leg_{leg_i}_reserve'].iloc[-1]
        g = df[f'leg_{leg_i}_integrity'].iloc[-1]
        t = df[f'leg_{leg_i}_thrust'].iloc[-1]
        starvation_pct = (df[f'leg_{leg_i}_reserve'] == 0).mean() * 100
        print(f'  Leg {leg_i}: reserve={r:.4f}  integrity={g:.4f}  thrust={t:.4f}  '
              f'ticks_at_zero_reserve={starvation_pct:.0f}%')

## 7. Diagnostic summary

In [ ]:
last = df.iloc[-1]

print('=== BOT SUMMARY ===')
print(f'Archetype:          {last["archetype"]}')
print(f'Ticks observed:     {len(df)}')
print(f'Age at last tick:   {last["age_ticks"]} ticks')
print(f'Resources collected:{last["resources_collected"]:.2f}')
print(f'Distance moved:     {last["distance_moved"]:.1f} units')
print(f'Damage taken total: {last["damage_taken_total"]:.3f}')
print()

print('=== ORGAN INTEGRITY (final tick) ===')
for elem in ELEMENTS:
    val = last[f'organ_{elem}']
    bar = '█' * int(val // 5) + '░' * (20 - int(val // 5))
    print(f'  {elem:<6} {val:5.1f}  {bar}')
print()

print('=== STORAGE (final tick) ===')
for elem in ELEMENTS:
    print(f'  {elem:<6} {last[f"storage_{elem}"]:.3f}')
print()

print('=== BEHAVIOUR STATE BREAKDOWN ===')
state_counts = df['behavior_state'].value_counts()
for state, count in state_counts.items():
    pct = count / len(df) * 100
    print(f'  {state:<12} {count:>5} ticks  ({pct:.1f}%)')
print()

print('=== METABOLIC BALANCE ===')
# Iterate the organs that actually have a storage drain. This used to loop over all five
# and index a four-entry table, so it raised KeyError on WATER every time — the notebook
# never reached this far. Water is spent by the legs, not by an organ, and is reported
# separately below.
for elem in DRAIN_PER_TICK:
    intake  = df[f'intake_{elem}'].mean()
    demand  = DRAIN_PER_TICK[elem]
    deficit = (df[f'intake_{elem}'] < demand).mean() * 100
    print(f'  {elem:<6} mean_intake={intake:.5f}  drain={demand:.4f}  deficit_ticks={deficit:.0f}%')

water_spent = -df['storage_WATER'].diff().clip(upper=0).sum()
print(f'  WATER  spent by legs, not an organ drain: {water_spent:.3f} total '
      f'({water_spent / max(1, len(df)):.5f}/tick)')